<a href="https://colab.research.google.com/github/jsalafica/Data-Science-III/blob/master/Datos_Entrenamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

df = pd.read_csv(
    "/mnt/datos_entrenamiento.csv",
    sep=",",
    engine="python",
    on_bad_lines="skip",
    encoding="utf-8"
)

df.head(), df.columns


(                                            sintomas      servicios
 0  Dolor en el pecho irradiado al brazo izquierdo...    Cardiología
 1  Dolor precordial irradiado al brazo izquierdo ...    Hemodinamia
 2  Fiebre alta, tos persistente, dificultad respi...   Neumonología
 3  Fractura expuesta en pierna, herida abierta, d...  Traumatología
 4  Erupción cutánea con prurito, lesiones eritema...   Dermatología,
 Index(['sintomas', 'servicios'], dtype='object'))

In [4]:
# Ajustá si tus columnas se llaman distinto
COL_TEXTO = "sintomas"
COL_SERVICIOS = "servicios"

df = df[[COL_TEXTO, COL_SERVICIOS]].dropna()
df[COL_TEXTO] = df[COL_TEXTO].astype(str)
df[COL_SERVICIOS] = df[COL_SERVICIOS].astype(str)

df.shape


(348, 2)

In [5]:
import re

def primer_servicio(s):
    partes = [p.strip() for p in re.split(r"[;,]", s) if p.strip()]
    return partes[0] if partes else None

df["servicio"] = df[COL_SERVICIOS].apply(primer_servicio)
df = df.dropna(subset=["servicio"])

df["servicio"].value_counts()


,count
servicio,
Clínica Médica,69
Traumatología,35
Otorrinolaringología,32
Cirugía General,31
Neurocirugía,24
Obstetricia,22
Neonatología,20
Ginecología,16
Urología,16


In [6]:
import random

SINONIMOS = {
    "paciente": ["paciente", "el paciente"],
    "consulta": ["consulta", "acude", "ingresa"],
    "refiere": ["refiere", "manifiesta", "describe"],
    "presenta": ["presenta", "evidencia", "cursa con"],
    "dolor": ["dolor", "algia"],
    "fiebre": ["fiebre", "febrícula"],
    "vomitos": ["vómitos", "emesis"],
    "disnea": ["disnea", "dificultad respiratoria"],
    "tos": ["tos", "tos seca"],
}


In [7]:
def variar_texto(texto, prob=0.25):
    palabras = texto.split()
    out = []
    for w in palabras:
        wl = re.sub(r"[^\wáéíóúñü]", "", w.lower())
        if wl in SINONIMOS and random.random() < prob:
            out.append(random.choice(SINONIMOS[wl]))
        else:
            out.append(w)
    return " ".join(out)

def augmentar(df, objetivo=1000):
    filas = []
    base = df[[COL_TEXTO, "servicio"]].sample(frac=1, random_state=42)

    while len(filas) < objetivo:
        for _, r in base.iterrows():
            filas.append({
                "texto": variar_texto(r[COL_TEXTO]),
                "servicio": r["servicio"]
            })
            if len(filas) >= objetivo:
                break

    return pd.DataFrame(filas)


In [8]:
df_aug = augmentar(df, objetivo=1000)

df_aug.shape
df_aug["servicio"].value_counts()


,count
servicio,
Clínica Médica,202
Traumatología,100
Cirugía General,91
Otorrinolaringología,91
Neurocirugía,66
Obstetricia,64
Neonatología,59
Ginecología,46
Urología,44


In [9]:
df_aug.to_csv(
    "/mnt/dataset_singlelabel_1000.csv",
    index=False,
    sep=";",
    encoding="utf-8"
)

"/mnt/dataset_singlelabel_1000.csv"


'/mnt/dataset_singlelabel_1000.csv'